In [1]:
import sys
import os
%cd /media/mohamed-gaber/01D96E17175F8760/Artifitial_intelligence/GraduationProject

/media/mohamed-gaber/01D96E17175F8760/Artifitial_intelligence/GraduationProject


In [ ]:
from __future__ import annotations

# =============================================================================
# IMPORTS
# =============================================================================

import logging
import re
import traceback
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd
from langchain_core.documents import Document
from neo4j import GraphDatabase

from src.Chunking import get_chunks
from src.Config import get_settings
from src.Utils import (
    Amendment,
    ExtractedLaw,
    Schedule,
    ScheduleEntry,
    _stable_id,
    _to_western_digits,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
)
logger = logging.getLogger(__name__)

/media/mohamed-gaber/01D96E17175F8760/Artifitial_intelligence/GraduationProject/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# =============================================================================
# HELPERS
# =============================================================================

def _dedup_items(
    items: List[Dict[str, Any]], key_fields: List[str]
) -> List[Dict[str, Any]]:
    seen, out = set(), []
    for item in items:
        key = tuple(item.get(f) for f in key_fields)
        if key not in seen:
            seen.add(key)
            out.append(item)
    return out


def _normalize_arabic_text(text: str) -> str:
    if not text:
        return ""
    text = re.sub(r"[\u0617-\u061A\u064B-\u0652\u0670]", "", text)
    text = text.replace("\u0640", "")
    for old, new in [("أ","ا"),("إ","ا"),("آ","ا"),("ى","ي"),
                     ("ئ","ي"),("ة","ه"),("ؤ","و")]:
        text = text.replace(old, new)
    return re.sub(r"\s+", " ", text).strip()

In [4]:
# =============================================================================
# SCHEDULE EXTRACTORS
# =============================================================================

class DrugScheduleExtractor:
    SCHEDULE_HEADER_RE = re.compile(
        r'(?:الجدول|جدول)\s+رقم\s*\(?([0-9٠-٩]+)\)?', re.UNICODE)

    def __init__(self, raw_text: str):
        self.raw_text = raw_text

    def extract_schedules(self) -> List[Schedule]:
        schedules, matches = [], list(self.SCHEDULE_HEADER_RE.finditer(self.raw_text))
        if not matches:
            return []
        for i, match in enumerate(matches):
            schedule_num  = _to_western_digits(match.group(1))
            start         = match.end()
            end           = matches[i+1].start() if i+1 < len(matches) else len(self.raw_text)
            schedule_text = self.raw_text[start:end].strip()
            lines         = [l.strip() for l in schedule_text.split('\n') if l.strip()]
            schedules.append(Schedule(
                schedule_id     = f"drug_schedule_{schedule_num}",
                schedule_number = schedule_num,
                title           = lines[0] if lines else f"جدول رقم {schedule_num}",
                entries         = self._extract_entries(schedule_text),
            ))
        return schedules

    def _extract_entries(self, text: str) -> List[ScheduleEntry]:
        entries, sections = [], re.split(r'\n(\d+)\s*[–\-]\s*', text)
        for i in range(1, len(sections), 2):
            if i + 1 >= len(sections):
                break
            entry_num   = sections[i].strip()
            entry_text  = sections[i+1].strip()
            arabic_name = english_name = chemical_name = trade_names = None
            nm = re.match(r'^([^(]+)(?:\(([^)]+)\))?', entry_text.split('\n')[0])
            if nm:
                arabic_name  = nm.group(1).strip()
                english_name = nm.group(2).strip() if nm.group(2) else None
            cm = re.search(
                r'الاسم الكيميائي:\s*(.+?)(?=\n|Chemical Name:|الوصف:|الأسماء|$)',
                entry_text, re.UNICODE)
            chemical_name = cm.group(1).strip() if cm else None
            tm = re.search(
                r'الأسماء التجارية:\s*(.+?)(?=\n---|\n\d+|$)', entry_text, re.UNICODE)
            if tm:
                trade_names = [n.strip() for n in re.split(r'[،,\-–]', tm.group(1))
                               if n.strip()]
            entries.append(ScheduleEntry(
                entry_number  = entry_num,
                arabic_name   = arabic_name,
                english_name  = english_name,
                chemical_name = chemical_name,
                trade_names   = trade_names,
            ))
        return entries


class WeaponScheduleExtractor:
    SCHEDULE_HEADER_RE = re.compile(
        r'جدول\s+رقم\s*\(?([0-9٠-٩]+)\)?', re.UNICODE)

    def __init__(self, raw_text: str):
        self.raw_text = raw_text

    def extract_schedules(self) -> List[Schedule]:
        schedules, matches = [], list(self.SCHEDULE_HEADER_RE.finditer(self.raw_text))
        if not matches:
            return []
        for i, match in enumerate(matches):
            schedule_num  = _to_western_digits(match.group(1))
            start         = match.end()
            end           = matches[i+1].start() if i+1 < len(matches) else len(self.raw_text)
            schedule_text = self.raw_text[start:end].strip()
            lines         = [l.strip() for l in schedule_text.split('\n') if l.strip()]
            schedules.append(Schedule(
                schedule_id     = f"weapon_schedule_{schedule_num}",
                schedule_number = schedule_num,
                title           = lines[0] if lines else f"جدول رقم {schedule_num}",
                entries         = self._extract_entries(schedule_text),
            ))
        return schedules

    def _extract_entries(self, text: str) -> List[ScheduleEntry]:
        entries = []
        for line in text.split('\n'):
            m = re.match(r'^(\d+)\s*[–\-]\s*(.+)$', line.strip())
            if m:
                entries.append(ScheduleEntry(
                    entry_number = m.group(1),
                    arabic_name  = m.group(2).strip(),
                    description  = m.group(2).strip(),
                ))
        return entries

In [5]:
# =============================================================================
# BASE LAW EXTRACTOR
# =============================================================================

class EnhancedBaseLawExtractor:

    ARTICLE_MARK_RE = re.compile(
        r"""(?m)^\s*(?:المادة|مادة)\s*(?:\(\s*)?(?P<num>[0-9٠-٩]+(?:\s*(?:مكرر(?:[اأإآى])?)?)?(?:\s*\(\s*[اأإآA-Za-z]\s*\))?)(?:\s*\))?
\s*[:：\-–\s](?P<rest>.*)$""", re.VERBOSE)

    REF_RE = re.compile(
        r"""(?:المادة|مادة|المواد)\s*(?:\(\s*)?(?P<num>[0-9٠-٩]+(?:\s*(?:مكرر(?:[اأإآى])?)?)?(?:\s*\(\s*[اأإآA-Za-z]\s*\))?)(?:\s*\))?(?:\s*(?:و|،|,)\s*(?:\(\s*)?(?P<num2>[0-9٠-٩]+(?:\s*مكرر(?:[اأإآى])?)?)(?:\s*\))?)?""",
        re.VERBOSE)

    PENALTY_PATTERNS = {
        'سجن': re.compile(
            r'(?:السجن|بالسجن|سجن[اً]?)\s*(?:مدة|لمدة)?\s*(?:(?:من|لا\s+تقل\s+عن)\s*)?([0-9٠-٩]+)\s*(?:إلى|حتى|-)\s*([0-9٠-٩]+)\s*(سنة|سنوات|عام|أعوام)',
            re.UNICODE),
        'غرامة': re.compile(
            r'(?:غرامة|بغرامة)\s*(?:مالية)?\s*(?:لا\s+)?(?:تقل\s+عن|من)\s*([0-9٠-٩,]+)\s*(?:جنيه|دولار|ريال)?(?:\s*(?:ولا\s+تزيد\s+على|إلى|حتى|-)\s*([0-9٠-٩,]+))?',
            re.UNICODE),
        'إعدام':      re.compile(r'(?:الإعدام|بالإعدام|القتل|قتل[اً]?|إعدام[اً]?)', re.UNICODE),
        'أشغال_شاقة': re.compile(r'الأشغال\s+الشاقة', re.UNICODE),
    }

    DEF_RE = re.compile(
        r'(?:يُقصد|المقصود|يُراد|تعني|يعني|يُعرَّف)\s+(?:ب|من)?\s*["\']?([^"\'"]+)["\']?\s*[:،]\s*(.+?)(?:\.|$)',
        re.UNICODE)

    _ARABIC_INDIC_TRANS = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")

    def __init__(self, raw_text: str, law_id: str, law_name: str,
                 promulgation_date: Optional[str] = None,
                 source: Optional[str] = None):
        self.raw_text           = raw_text or ""
        self._law_id            = law_id
        self._law_name          = law_name
        self._promulgation_date = promulgation_date
        self._source            = source or "الجريدة الرسمية"
        self._validation_warnings: List[str] = []

    def _to_western_digits(self, s: str) -> str:
        return s.translate(self._ARABIC_INDIC_TRANS)

    def _normalize_article_no(self, token: str) -> str:
        token = self._to_western_digits(token.strip())
        token = re.sub(r"\s+", " ", token)
        token = re.sub(r"مكر(?:ر[اأإآىًٍَُِّْ]?)", "مكرر", token)
        for old, new in [("أولاً","أولا"),("أولا","أولا"),
                         ("ثانياً","ثانيا"),("ثانيا","ثانيا")]:
            token = token.replace(old, new)
        return token.strip()

    def extract_law_meta(self) -> Dict[str, Any]:
        return {
            "law_id":              self._law_id,
            "title":               self._law_name,
            "promulgation_date":   self._promulgation_date,
            "effective_date":      self._promulgation_date,
            "source":              self._source,
            "language":            "ar",
            "text_length":         len(self.raw_text),
            "validation_warnings": self._validation_warnings,
        }

    def extract_articles(self) -> List[Dict[str, str]]:
        text    = self.raw_text
        matches = list(self.ARTICLE_MARK_RE.finditer(text))
        articles: List[Dict[str, str]] = []
        if not matches:
            return [{"article_number": None, "text": text.strip(), "language": "ar"}]
        if matches[0].start() > 0:
            preamble = text[:matches[0].start()].strip()
            if preamble:
                articles.append({"article_number": None, "text": preamble, "language": "ar"})
        for i, m in enumerate(matches):
            start      = m.end()
            end        = matches[i+1].start() if i+1 < len(matches) else len(text)
            body       = text[start:end].strip()
            article_no = self._normalize_article_no(m.group("num") or "")
            articles.append({"article_number": article_no, "text": body, "language": "ar"})
        return articles

    def extract_penalties(self, articles: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        penalties = []
        for article in articles:
            article_no = article.get("article_number")
            if not article_no:
                continue
            text = article.get("text", "")
            for m in self.PENALTY_PATTERNS['سجن'].finditer(text):
                penalties.append({
                    "article_number": article_no, "penalty_type": "سجن",
                    "min_value": int(self._to_western_digits(m.group(1))),
                    "max_value": int(self._to_western_digits(m.group(2))),
                    "unit": "سنة"})
            for m in self.PENALTY_PATTERNS['غرامة'].finditer(text):
                min_v = self._to_western_digits(m.group(1).replace(',', ''))
                max_v = m.group(2)
                penalties.append({
                    "article_number": article_no, "penalty_type": "غرامة",
                    "min_value": int(min_v) if min_v else None,
                    "max_value": int(self._to_western_digits(
                        max_v.replace(',', ''))) if max_v else None,
                    "unit": "جنيه"})
            if self.PENALTY_PATTERNS['إعدام'].search(text):
                penalties.append({"article_number": article_no, "penalty_type": "إعدام",
                                   "min_value": None, "max_value": None, "unit": None})
            if self.PENALTY_PATTERNS['أشغال_شاقة'].search(text):
                penalties.append({"article_number": article_no, "penalty_type": "أشغال_شاقة",
                                   "min_value": None, "max_value": None, "unit": None})
        return _dedup_items(penalties, ["article_number", "penalty_type", "min_value", "max_value"])

    def extract_definitions(self, articles: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        definitions = []
        for article in articles:
            article_no = article.get("article_number")
            if not article_no:
                continue
            for m in self.DEF_RE.finditer(article.get("text", "")):
                definitions.append({
                    "term":               m.group(1).strip(),
                    "definition_text":    m.group(2).strip(),
                    "defined_in_article": article_no,
                    "language":           "ar",
                })
        return _dedup_items(definitions, ["term", "defined_in_article"])

    def extract_references(self, articles: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        refs = set()
        for a in articles:
            from_no = str(a.get("article_number", "")).strip()
            if not from_no or from_no == "None":
                continue
            for m in self.REF_RE.finditer(a.get("text", "")):
                for to_no in [m.group("num"), m.group("num2")]:
                    if to_no:
                        to_no = self._normalize_article_no(to_no)
                        if to_no and to_no != from_no:
                            refs.add((from_no, to_no, "direct"))
        return [{"from_article": f, "to_article": t, "reference_type": rt}
                for f, t, rt in sorted(refs)]

    def extract_topics(self, articles: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        topics_map = {
            "قتل":    ["قتل", "قاتل"],
            "سرقة":   ["سرقة", "سارق"],
            "مخدرات": ["مخدر", "مخدرات"],
            "أسلحة":  ["سلاح", "أسلحة"],
            "إرهاب":  ["إرهاب", "إرهابي"],
        }
        article_topics = []
        for article in articles:
            article_no = article.get("article_number")
            if not article_no:
                continue
            text = _normalize_arabic_text(article.get("text", ""))
            for topic, keywords in topics_map.items():
                if any(_normalize_arabic_text(kw) in text for kw in keywords):
                    article_topics.append({
                        "article_number": article_no,
                        "topic_name":     topic,
                        "confidence":     0.8,
                    })
                    break
        return _dedup_items(article_topics, ["article_number", "topic_name"])

    def extract_schedules(self) -> List[Schedule]:
        return []

    def extract_all(self) -> ExtractedLaw:
        meta     = self.extract_law_meta()
        articles = self.extract_articles()
        return ExtractedLaw(
            law_meta    = meta,
            articles    = articles,
            penalties   = self.extract_penalties(articles),
            definitions = self.extract_definitions(articles),
            references  = self.extract_references(articles),
            topics      = self.extract_topics(articles),
            amendments  = [],
            schedules   = self.extract_schedules(),
        )

In [6]:
# =============================================================================
# LAW-SPECIFIC EXTRACTORS
# =============================================================================

class PenalCodeExtractor(EnhancedBaseLawExtractor):
    def __init__(self, raw_text: str):
        super().__init__(raw_text, "penal_code", "قانون العقوبات", "1937-07-31")

class MoneyLaunderingExtractor(EnhancedBaseLawExtractor):
    def __init__(self, raw_text: str):
        super().__init__(raw_text, "money_laundering", "قانون مكافحة غسل الأموال", "2002-05-15")

class WeaponsAmmunitionExtractor(EnhancedBaseLawExtractor):
    def __init__(self, raw_text: str):
        super().__init__(raw_text, "weapons_ammunition", "قانون الأسلحة والذخيرة", "1954-09-22")
    def extract_schedules(self) -> List[Schedule]:
        return WeaponScheduleExtractor(self.raw_text).extract_schedules()

class CriminalConstitutionExtractor(EnhancedBaseLawExtractor):
    def __init__(self, raw_text: str):
        super().__init__(raw_text, "criminal_constitution", "الدستور الجنائي")

class DrugsLawExtractor(EnhancedBaseLawExtractor):
    def __init__(self, raw_text: str):
        super().__init__(raw_text, "anti_drugs", "قانون مكافحة المخدرات", "1960-05-01")
    def extract_schedules(self) -> List[Schedule]:
        return DrugScheduleExtractor(self.raw_text).extract_schedules()

class CriminalProcedureExtractor(EnhancedBaseLawExtractor):
    def __init__(self, raw_text: str):
        super().__init__(raw_text, "criminal_procedure", "قانون الإجراءات الجنائية", "1950-10-18")

class AntiTerrorExtractor(EnhancedBaseLawExtractor):
    def __init__(self, raw_text: str):
        super().__init__(raw_text, "anti_terror", "قانون مكافحة الإرهاب", "2015-08-16")

class EmergencyLawExtractor(EnhancedBaseLawExtractor):
    def __init__(self, raw_text: str):
        super().__init__(raw_text, "emergency_law", "قانون الطوارئ", "1958-01-01")

class CybercrimeExtractor(EnhancedBaseLawExtractor):
    def __init__(self, raw_text: str):
        super().__init__(raw_text, "cybercrime", "قانون مكافحة جرائم تقنية المعلومات", "2018-07-17")


def get_extractor(law_key: str, raw_text: str) -> EnhancedBaseLawExtractor:
    mapping = {
        "penal_code":            PenalCodeExtractor,
        "money_laundering":      MoneyLaunderingExtractor,
        "weapons_ammunition":    WeaponsAmmunitionExtractor,
        "criminal_constitution": CriminalConstitutionExtractor,
        "anti_drugs":            DrugsLawExtractor,
        "criminal_procedure":    CriminalProcedureExtractor,
        "anti_terror":           AntiTerrorExtractor,
        "emergency_law":         EmergencyLawExtractor,
        "cybercrime":            CybercrimeExtractor,
    }
    if law_key not in mapping:
        raise KeyError(f"Unknown law_key: '{law_key}'")
    return mapping[law_key](raw_text)

In [7]:
# =============================================================================
# CHUNK-BASED INGESTION
# =============================================================================

def ingest_dataset(chunks: List[Document]) -> List[Dict]:
    """
    Group article chunks by law_id.
    Articles come directly from chunk metadata — no re-parsing needed.
    The extractor is used only for penalties, definitions, references,
    topics, and schedules (which operate on the article list, not raw text).
    Each summary includes ``articles`` (the list itself) for the pipeline.
    """
    law_data: Dict[str, Dict] = {}
    for chunk in chunks:
        if chunk.metadata.get("chunk_type") != "article":
            continue
        law_key = chunk.metadata["law_id"]
        if law_key not in law_data:
            law_data[law_key] = {
                "law_title": chunk.metadata.get("law_title", law_key),
                "articles":  [],
            }
        law_data[law_key]["articles"].append({
            "article_number": chunk.metadata.get("article_number"),
            "text":           chunk.page_content,
            "language":       "ar",
        })

    summaries = []
    for law_key, data in law_data.items():
        articles = data["articles"]
        try:
            if not articles:
                summaries.append({"law_title": data["law_title"], "law_key": law_key,
                                  "articles": [], "error": "empty"})
                continue
            # Use extractor only for NLP-based extraction (penalties, refs, etc.)
            # Pass a dummy empty string — we override articles immediately after
            extractor        = get_extractor(law_key, "")
            extractor.raw_text = "\n".join(a["text"] for a in articles)
            penalties   = extractor.extract_penalties(articles)
            definitions = extractor.extract_definitions(articles)
            references  = extractor.extract_references(articles)
            topics      = extractor.extract_topics(articles)
            schedules   = extractor.extract_schedules()
            law_meta    = extractor.extract_law_meta()
            summaries.append({
                "law_title":   data["law_title"],
                "law_key":     law_key,
                "law_id":      law_meta.get("law_id"),
                "articles":    articles,           # list, not count — pipeline uses this
                "penalties":   penalties,
                "definitions": definitions,
                "references":  references,
                "topics":      topics,
                "schedules":   schedules,
                "law_meta":    law_meta,
                "error":       None,
            })
        except Exception as e:
            summaries.append({"law_title": data["law_title"], "law_key": law_key,
                               "articles": [], "error": str(e)})
    return summaries


def ingest_amendment_files(chunks: List[Document]) -> Dict[str, List[Amendment]]:
    """
    Group amended_article chunks by (law_id, amendment_law_number, amendment_date).
    Builds Amendment objects directly from chunk metadata — no file reading needed.
    amendment_type from chunk metadata is authoritative.
    """
    groups: Dict[tuple, Dict] = {}
    for chunk in chunks:
        if chunk.metadata.get("chunk_type") != "amended_article":
            continue
        m          = chunk.metadata
        law_key    = m["law_id"]
        law_num    = m.get("amendment_law_number", "unknown")
        amend_date = m.get("amendment_date",       "unknown")
        amend_type = m.get("amendment_type",       "modification")
        article_no = m.get("article_number")
        key        = (law_key, law_num, amend_date)
        if key not in groups:
            groups[key] = {
                "law_key":    law_key,
                "law_num":    law_num,
                "amend_date": amend_date,
                "amend_type": amend_type,
                "articles":   [],
                "texts":      [],
            }
        if article_no and article_no not in groups[key]["articles"]:
            groups[key]["articles"].append(article_no)
        groups[key]["texts"].append(chunk.page_content)

    amendments_by_law: Dict[str, List[Amendment]] = {}
    for (law_key, law_num, amend_date), data in groups.items():
        if not data["articles"]:
            logger.warning(f"  [SKIP] {law_key} {law_num}/{amend_date} — no article numbers")
            continue
        year         = amend_date[:4] if amend_date != "unknown" else "????"
        amendment_id = _stable_id(law_key, law_num, year)
        full_text    = "\n".join(data["texts"])
        desc_m       = re.search(
            r'المادة الأولى\s*[:：]?\s*(.*?)(?=المادة الثانية|$)',
            full_text, re.DOTALL)
        description  = desc_m.group(1).strip()[:500] if desc_m else full_text[:500]

        amendment = Amendment(
            amendment_id            = amendment_id,
            amendment_law_number    = law_num,
            amendment_date          = amend_date,
            amendment_law_title     = f"قانون رقم {law_num} لسنة {year}",
            amended_article_numbers = data["articles"],
            amendment_type          = data["amend_type"],
            description             = description,
            effective_date          = amend_date,
        )
        amendments_by_law.setdefault(law_key, []).append(amendment)
        logger.info(f"  ✓ {law_key} law={law_num}/{year} "
                    f"type={data['amend_type']} articles={data['articles']}")

    return amendments_by_law

In [8]:
# =============================================================================
# KNOWLEDGE GRAPH
# =============================================================================

class EnhancedLegalKnowledgeGraph:

    def __init__(self, uri: str, user: str, password: str):
        self.driver = GraphDatabase.driver(uri, auth=(user, password))
        try:
            self.driver.verify_connectivity()
            logger.info("✓ Neo4j connected")
        except Exception as e:
            logger.error(f"Connection failed: {e}"); raise

    def close(self):
        self.driver.close()

    def setup_schema(self, drop_existing: bool = False):
        with self.driver.session() as session:
            if drop_existing:
                session.run("MATCH (n) DETACH DELETE n")
                logger.info("✓ Existing data cleared")
            constraints = [
                "CREATE CONSTRAINT law_id_unique        IF NOT EXISTS FOR (l:Law)          REQUIRE l.law_id IS UNIQUE",
                "CREATE CONSTRAINT article_id_unique    IF NOT EXISTS FOR (a:Article)       REQUIRE a.article_id IS UNIQUE",
                "CREATE CONSTRAINT penalty_id_unique    IF NOT EXISTS FOR (p:Penalty)       REQUIRE p.penalty_id IS UNIQUE",
                "CREATE CONSTRAINT definition_id_unique IF NOT EXISTS FOR (d:Definition)    REQUIRE d.definition_id IS UNIQUE",
                "CREATE CONSTRAINT topic_id_unique      IF NOT EXISTS FOR (t:Topic)         REQUIRE t.topic_id IS UNIQUE",
                "CREATE CONSTRAINT schedule_id_unique   IF NOT EXISTS FOR (s:Schedule)      REQUIRE s.schedule_id IS UNIQUE",
                "CREATE CONSTRAINT entry_id_unique      IF NOT EXISTS FOR (e:ScheduleEntry) REQUIRE e.entry_id IS UNIQUE",
                "CREATE CONSTRAINT substance_id_unique  IF NOT EXISTS FOR (sub:Substance)   REQUIRE sub.substance_id IS UNIQUE",
                "CREATE CONSTRAINT amendment_id_unique  IF NOT EXISTS FOR (am:Amendment)    REQUIRE am.amendment_id IS UNIQUE",
            ]
            indexes = [
                "CREATE INDEX supersedes_date_idx IF NOT EXISTS FOR ()-[r:SUPERSEDES]-() ON (r.amendment_date)",
            ]
            for stmt in constraints + indexes:
                try:
                    session.run(stmt)
                except Exception:
                    pass
            logger.info("✓ Schema ready")

    def create_law(self, law_meta: Dict[str, Any]):
        with self.driver.session() as session:
            session.run("""
                MERGE (l:Law {law_id: $law_id})
                SET l.title             = $title,
                    l.promulgation_date = $promulgation_date,
                    l.source            = $source,
                    l.language          = $language
            """, law_id=law_meta.get("law_id"),
                title=law_meta.get("title"),
                promulgation_date=law_meta.get("promulgation_date"),
                source=law_meta.get("source"),
                language=law_meta.get("language", "ar"))

    def create_article(self, law_id: str, article_number: Optional[str],
                       text: str, position: int):
        article_id = _stable_id(law_id, article_number or "preamble", position)
        with self.driver.session() as session:
            session.run("""
                MATCH (l:Law {law_id: $law_id})
                MERGE (a:Article {article_id: $article_id})
                SET a.article_number = $article_number,
                    a.text           = $text,
                    a.law_id         = $law_id
                MERGE (l)-[r:CONTAINS]->(a)
                SET r.position = $position
            """, law_id=law_id, article_id=article_id,
                article_number=article_number, text=text, position=position)

    def create_penalty(self, penalty: Dict[str, Any], law_id: str):
        penalty_id = _stable_id(law_id, penalty.get("article_number"),
                                penalty.get("penalty_type"))
        with self.driver.session() as session:
            session.run("""
                MATCH (a:Article {law_id: $law_id, article_number: $article_number})
                MERGE (p:Penalty {penalty_id: $penalty_id})
                SET p.penalty_type = $penalty_type,
                    p.min_value    = $min_value,
                    p.max_value    = $max_value,
                    p.unit         = $unit
                MERGE (a)-[:HAS_PENALTY]->(p)
            """, law_id=law_id, penalty_id=penalty_id,
                article_number=penalty.get("article_number"),
                penalty_type=penalty.get("penalty_type"),
                min_value=penalty.get("min_value"),
                max_value=penalty.get("max_value"),
                unit=penalty.get("unit"))

    def create_definition(self, definition: Dict[str, Any], law_id: str):
        def_id = _stable_id(law_id, definition.get("term"))
        with self.driver.session() as session:
            session.run("""
                MATCH (a:Article {law_id: $law_id, article_number: $article_number})
                MERGE (d:Definition {definition_id: $def_id})
                SET d.term            = $term,
                    d.definition_text = $definition_text
                MERGE (a)-[:DEFINES]->(d)
            """, law_id=law_id, def_id=def_id,
                article_number=definition.get("defined_in_article"),
                term=definition.get("term"),
                definition_text=definition.get("definition_text"))

    def create_topic(self, topic_name: str):
        with self.driver.session() as session:
            session.run("MERGE (t:Topic {topic_id: $id, name: $name})",
                        id=_stable_id(topic_name), name=topic_name)

    def link_article_to_topic(self, law_id: str, article_number: str,
                               topic_name: str, confidence: float):
        with self.driver.session() as session:
            session.run("""
                MATCH (a:Article {law_id: $law_id, article_number: $article_number})
                MATCH (t:Topic {name: $topic_name})
                MERGE (a)-[r:TAGGED_WITH]->(t)
                SET r.confidence = $confidence
            """, law_id=law_id, article_number=article_number,
                topic_name=topic_name, confidence=confidence)

    def create_article_reference(self, law_id: str, from_article: str,
                                  to_article: str, ref_type: str):
        with self.driver.session() as session:
            session.run("""
                MATCH (from:Article {law_id: $law_id, article_number: $from_article})
                MATCH (to:Article   {law_id: $law_id, article_number: $to_article})
                MERGE (from)-[r:REFERENCES]->(to)
                SET r.reference_type = $ref_type
            """, law_id=law_id, from_article=from_article,
                to_article=to_article, ref_type=ref_type)

    def create_schedule(self, law_id: str, schedule: Schedule):
        with self.driver.session() as session:
            session.run("""
                MATCH (l:Law {law_id: $law_id})
                MERGE (s:Schedule {schedule_id: $schedule_id})
                SET s.schedule_number = $schedule_number,
                    s.title           = $title,
                    s.entry_count     = $count
                MERGE (l)-[:HAS_SCHEDULE]->(s)
            """, law_id=law_id, schedule_id=schedule.schedule_id,
                schedule_number=schedule.schedule_number,
                title=schedule.title, count=len(schedule.entries))

    def create_schedule_entry(self, schedule_id: str,
                               entry: ScheduleEntry, position: int):
        entry_id = _stable_id(schedule_id, entry.entry_number, position)
        with self.driver.session() as session:
            session.run("""
                MATCH (s:Schedule {schedule_id: $schedule_id})
                MERGE (e:ScheduleEntry {entry_id: $entry_id})
                SET e.entry_number = $entry_number,
                    e.arabic_name  = $arabic_name,
                    e.english_name = $english_name,
                    e.description  = $description
                MERGE (s)-[r:CONTAINS_ENTRY]->(e)
                SET r.position = $position
            """, schedule_id=schedule_id, entry_id=entry_id,
                entry_number=entry.entry_number, arabic_name=entry.arabic_name,
                english_name=entry.english_name, description=entry.description,
                position=position)
        if entry.chemical_name:
            self.create_substance(entry_id, entry)

    def create_substance(self, entry_id: str, entry: ScheduleEntry):
        sub_id = _stable_id("substance", entry.arabic_name or entry.english_name)
        with self.driver.session() as session:
            session.run("""
                MATCH (e:ScheduleEntry {entry_id: $entry_id})
                MERGE (sub:Substance {substance_id: $substance_id})
                SET sub.arabic_name   = $arabic_name,
                    sub.english_name  = $english_name,
                    sub.chemical_name = $chemical_name,
                    sub.trade_names   = $trade_names
                MERGE (e)-[:DEFINES_SUBSTANCE]->(sub)
            """, entry_id=entry_id, substance_id=sub_id,
                arabic_name=entry.arabic_name, english_name=entry.english_name,
                chemical_name=entry.chemical_name,
                trade_names=entry.trade_names or [])

    def create_amendment(self, law_id: str, amendment: Amendment):
        desc = (amendment.description or "")[:500]
        with self.driver.session() as session:
            session.run("""
                MATCH (l:Law {law_id: $law_id})
                MERGE (am:Amendment {amendment_id: $amendment_id})
                SET am.amendment_law_number = $amendment_law_number,
                    am.amendment_law_title  = $amendment_law_title,
                    am.amendment_date       = $amendment_date,
                    am.amendment_type       = $amendment_type,
                    am.description          = $description,
                    am.effective_date       = $effective_date
                MERGE (l)-[:HAS_AMENDMENT]->(am)
            """, law_id=law_id,
                amendment_id=amendment.amendment_id,
                amendment_law_number=amendment.amendment_law_number,
                amendment_law_title=amendment.amendment_law_title,
                amendment_date=amendment.amendment_date,
                amendment_type=amendment.amendment_type,
                description=desc,
                effective_date=amendment.effective_date)

        for article_number in amendment.amended_article_numbers:
            if not article_number:
                continue
            if amendment.amendment_type == 'addition':
                new_article_id = _stable_id(law_id, article_number,
                                            "added", amendment.amendment_date)
                with self.driver.session() as session:
                    session.run("""
                        MATCH (l:Law {law_id: $law_id})
                        MATCH (am:Amendment {amendment_id: $amendment_id})
                        MERGE (a_new:Article {article_id: $new_article_id})
                        ON CREATE SET
                            a_new.article_number = $article_number,
                            a_new.law_id         = $law_id,
                            a_new.text           = $description,
                            a_new.version        = $amendment_date,
                            a_new.is_addition    = true
                        MERGE (l)-[:CONTAINS]->(a_new)
                        MERGE (a_new)-[:AMENDED_BY {
                            amendment_type: $amendment_type,
                            amendment_date: $amendment_date
                        }]->(am)
                    """, law_id=law_id,
                        amendment_id=amendment.amendment_id,
                        new_article_id=new_article_id,
                        article_number=article_number,
                        description=desc,
                        amendment_date=amendment.amendment_date,
                        amendment_type=amendment.amendment_type)
                    session.run("""
                        MATCH (a_new:Article {article_id: $new_article_id})
                        MATCH (a_old:Article {law_id: $law_id,
                                              article_number: $article_number})
                        WHERE a_old.article_id <> $new_article_id
                          AND a_old.is_addition IS NULL
                        MERGE (a_new)-[:SUPERSEDES {
                            amendment_id:   $amendment_id,
                            amendment_date: $amendment_date
                        }]->(a_old)
                    """, new_article_id=new_article_id, law_id=law_id,
                        article_number=article_number,
                        amendment_id=amendment.amendment_id,
                        amendment_date=amendment.amendment_date)
            else:
                new_article_id = _stable_id(law_id, article_number,
                                            "amended", amendment.amendment_date)
                with self.driver.session() as session:
                    session.run("""
                        MATCH (am:Amendment {amendment_id: $amendment_id})
                        MATCH (a:Article {law_id: $law_id,
                                          article_number: $article_number})
                        MERGE (a)-[r:AMENDED_BY]->(am)
                        SET r.amendment_type = $amendment_type,
                            r.amendment_date = $amendment_date
                    """, amendment_id=amendment.amendment_id, law_id=law_id,
                        article_number=article_number,
                        amendment_type=amendment.amendment_type,
                        amendment_date=amendment.amendment_date)
                with self.driver.session() as session:
                    session.run("""
                        MATCH (l:Law {law_id: $law_id})
                        MATCH (am:Amendment {amendment_id: $amendment_id})
                        MATCH (a_old:Article {law_id: $law_id,
                                              article_number: $article_number})
                        WHERE a_old.article_id <> $new_article_id
                          AND a_old.is_amended IS NULL
                        MERGE (a_new:Article {article_id: $new_article_id})
                        ON CREATE SET
                            a_new.article_number = $article_number,
                            a_new.law_id         = $law_id,
                            a_new.text           = $description,
                            a_new.version        = $amendment_date,
                            a_new.is_amended     = true
                        MERGE (l)-[:CONTAINS]->(a_new)
                        MERGE (a_new)-[:SUPERSEDES {
                            amendment_id:   $amendment_id,
                            amendment_date: $amendment_date
                        }]->(a_old)
                    """, law_id=law_id,
                        amendment_id=amendment.amendment_id,
                        article_number=article_number,
                        new_article_id=new_article_id,
                        description=desc,
                        amendment_date=amendment.amendment_date)

    def import_law(self, summary: Dict, verbose: bool = True) -> Dict[str, int]:
        """
        Import one law from an ``ingest_dataset`` summary dict.
        All data is already extracted — no re-parsing.
        """
        stats    = {"articles": 0, "penalties": 0, "definitions": 0,
                    "references": 0, "topics": 0, "schedules": 0,
                    "entries": 0, "substances": 0}
        law_meta = summary["law_meta"]
        law_id   = law_meta.get("law_id")
        if verbose:
            print(f"\n{'='*70}\nIMPORTING: {law_meta.get('title')}\n{'='*70}")
        self.create_law(law_meta)
        for i, article in enumerate(summary["articles"]):
            self.create_article(law_id, article.get("article_number"),
                                article.get("text", ""), i)
            stats["articles"] += 1
        for penalty in summary["penalties"]:
            self.create_penalty(penalty, law_id)
            stats["penalties"] += 1
        for definition in summary["definitions"]:
            self.create_definition(definition, law_id)
            stats["definitions"] += 1
        for topic_name in set(t["topic_name"] for t in summary["topics"]):
            self.create_topic(topic_name)
        for topic in summary["topics"]:
            self.link_article_to_topic(law_id, topic["article_number"],
                                       topic["topic_name"], topic.get("confidence", 1.0))
            stats["topics"] += 1
        for ref in summary["references"]:
            self.create_article_reference(law_id, ref.get("from_article"),
                                          ref.get("to_article"),
                                          ref.get("reference_type", "direct"))
            stats["references"] += 1
        for schedule in summary["schedules"]:
            self.create_schedule(law_id, schedule)
            stats["schedules"] += 1
            for i, entry in enumerate(schedule.entries):
                self.create_schedule_entry(schedule.schedule_id, entry, i)
                stats["entries"] += 1
                if entry.chemical_name:
                    stats["substances"] += 1
        if verbose:
            print(f"  ✓ {stats['articles']} articles | {stats['penalties']} penalties "
                  f"| {stats['schedules']} schedules")
        return stats

    def get_statistics(self) -> Dict[str, Any]:
        with self.driver.session() as session:
            stats: Dict[str, Any] = {"node_counts": {}, "relationship_counts": {}}
            for label in ["Law", "Article", "Penalty", "Definition", "Topic",
                          "Schedule", "ScheduleEntry", "Substance", "Amendment"]:
                result = session.run(f"MATCH (n:{label}) RETURN count(n) AS count")
                stats["node_counts"][label] = result.single()["count"]
            for rel in ["CONTAINS", "REFERENCES", "HAS_PENALTY", "DEFINES",
                        "TAGGED_WITH", "HAS_SCHEDULE", "CONTAINS_ENTRY",
                        "DEFINES_SUBSTANCE", "HAS_AMENDMENT", "AMENDED_BY", "SUPERSEDES"]:
                result = session.run(f"MATCH ()-[r:{rel}]->() RETURN count(r) AS count")
                stats["relationship_counts"][rel] = result.single()["count"]
            return stats

    def print_statistics(self):
        stats = self.get_statistics()
        print("\n" + "="*70 + "\nKNOWLEDGE GRAPH STATISTICS\n" + "="*70)
        print("\nNodes:")
        for label, count in stats["node_counts"].items():
            print(f"  {label}: {count}")
        print("\nRelationships:")
        for rel, count in stats["relationship_counts"].items():
            print(f"  {rel}: {count}")
        print("\n" + "="*70)

    def query_amendments(self, law_id: Optional[str] = None) -> List[Dict]:
        with self.driver.session() as session:
            if law_id:
                result = session.run("""
                    MATCH (l:Law {law_id: $law_id})-[:HAS_AMENDMENT]->(am:Amendment)
                    OPTIONAL MATCH (a:Article)-[:AMENDED_BY]->(am)
                    RETURN am, collect(DISTINCT a.article_number) AS affected_articles
                    ORDER BY am.amendment_date
                """, law_id=law_id)
            else:
                result = session.run("""
                    MATCH (l:Law)-[:HAS_AMENDMENT]->(am:Amendment)
                    OPTIONAL MATCH (a:Article)-[:AMENDED_BY]->(am)
                    RETURN l.law_id AS law_id, l.title AS law_title,
                           am, collect(DISTINCT a.article_number) AS affected_articles
                    ORDER BY am.amendment_date
                """)
            amendments = []
            for record in result:
                amendment_data = dict(record["am"])
                amendment_data["affected_articles"] = record["affected_articles"]
                if not law_id:
                    amendment_data["law_id"]    = record["law_id"]
                    amendment_data["law_title"] = record["law_title"]
                amendments.append(amendment_data)
            return amendments

    def query_article_history(self, law_id: str, article_number: str) -> List[Dict]:
        with self.driver.session() as session:
            result = session.run("""
                MATCH (a:Article {law_id: $law_id, article_number: $article_number})
                OPTIONAL MATCH (a)-[:AMENDED_BY]->(am:Amendment)
                OPTIONAL MATCH (a_new:Article)-[:SUPERSEDES]->(a)
                OPTIONAL MATCH (a)-[:SUPERSEDES]->(a_old:Article)
                RETURN
                    a.article_number                   AS article,
                    a.version                          AS version,
                    a.is_amended                       AS is_amended,
                    a.is_addition                      AS is_addition,
                    collect(DISTINCT {
                        amendment_id: am.amendment_id,
                        law_num:      am.amendment_law_number,
                        date:         am.amendment_date,
                        type:         am.amendment_type
                    })                                 AS amendments,
                    collect(DISTINCT a_new.article_id) AS superseded_by,
                    collect(DISTINCT a_old.article_id) AS supersedes
                ORDER BY a.version
            """, law_id=law_id, article_number=article_number)
            return [dict(r) for r in result]

In [9]:
# =============================================================================
# PIPELINE
# =============================================================================

def build_knowledge_graph(
    neo4j_uri:      str,
    neo4j_user:     str,
    neo4j_password: str,
    drop_existing:  bool = True,
    verbose:        bool = True,
) -> EnhancedLegalKnowledgeGraph:
    """
    Full pipeline. Calls ``get_chunks()`` internally — no paths or
    folder mappings needed.

    Phase 0 — chunk all law files once.
    Phase 1 — extract and summarise all laws.
    Phase 2 — import law nodes + articles into Neo4j.
    Phase 3 — import amendments from amended_article chunks into Neo4j.
    """
    # ── Phase 0: chunk ────────────────────────────────────────────────────
    print("\n" + "="*70 + "\nPHASE 0: CHUNKING\n" + "="*70)
    chunks = get_chunks()
    print(f"  Total chunks: {len(chunks):,}")

    # ── Phase 1: extraction summary ───────────────────────────────────────
    print("\n" + "="*70 + "\nPHASE 1: EXTRACTION SUMMARY\n" + "="*70)
    summaries  = ingest_dataset(chunks)
    # Display counts only (skip list fields)
    _SKIP = {"articles", "penalties", "definitions", "references",
             "topics", "schedules", "law_meta"}
    df_rows = []
    for s in summaries:
        row = {k: v for k, v in s.items() if k not in _SKIP}
        row["articles"]    = len(s.get("articles",    []))
        row["penalties"]   = len(s.get("penalties",   []))
        row["definitions"] = len(s.get("definitions", []))
        row["references"]  = len(s.get("references",  []))
        row["topics"]      = len(s.get("topics",      []))
        row["schedules"]   = len(s.get("schedules",   []))
        df_rows.append(row)
    print(pd.DataFrame(df_rows).to_string())
    successful = [s for s in summaries if s.get("error") is None]
    print(f"\n  ✓ {len(successful)}/{len(summaries)} laws extracted successfully")

    # ── Phase 2: build graph ──────────────────────────────────────────────
    print("\n" + "="*70 + "\nPHASE 2: IMPORTING LAWS\n" + "="*70)
    graph = EnhancedLegalKnowledgeGraph(neo4j_uri, neo4j_user, neo4j_password)
    graph.setup_schema(drop_existing=drop_existing)

    for summary in successful:
        if not summary.get("articles"):
            logger.warning(f"  [SKIP] {summary['law_key']} — no articles")
            continue
        graph.import_law(summary, verbose=verbose)

    # ── Phase 3: amendments ───────────────────────────────────────────────
    print("\n" + "="*70 + "\nPHASE 3: IMPORTING AMENDMENTS\n" + "="*70)
    amendments_by_law = ingest_amendment_files(chunks)
    total = 0
    for law_id, amendments in amendments_by_law.items():
        if verbose:
            print(f"\n  → {law_id}: {len(amendments)} amendment(s)")
        for amendment in amendments:
            try:
                graph.create_amendment(law_id, amendment)
                total += 1
                if verbose:
                    print(f"    ✓ law={amendment.amendment_law_number}/"
                          f"{amendment.amendment_date[:4]} "
                          f"| type={amendment.amendment_type} "
                          f"| articles={amendment.amended_article_numbers}")
            except Exception as e:
                logger.error(f"    ✗ {amendment.amendment_id}: {e}")
                if verbose:
                    traceback.print_exc()
    print(f"\n  ✓ Total amendments imported: {total}")

    print("\n✅ BUILD COMPLETE")
    graph.print_statistics()
    return graph

In [10]:
# =============================================================================
# RUN
# =============================================================================

graph = build_knowledge_graph(
    neo4j_uri      = get_settings().NEO4J_URI,
    neo4j_user     = get_settings().NEO4J_USERNAME,
    neo4j_password = get_settings().NEO4J_PASSWORD,
    drop_existing  = True,
    verbose        = True,
)
graph.close()

2026-03-09 07:24:37,924 - INFO -   3okobat.txt [article] → 538 chunks
2026-03-09 07:24:37,925 - INFO -   new_3okobat_num36_2020.txt [amendment] → 1 chunks
2026-03-09 07:24:37,926 - INFO -   new_3okobat_num6_2020.txt [amendment] → 2 chunks
2026-03-09 07:24:37,927 - INFO -   قانون-مكافحة-غسل-الامول_ultra_clean.txt [article] → 21 chunks
2026-03-09 07:24:37,928 - INFO -   asle7a.txt [article] → 52 chunks
2026-03-09 07:24:37,928 - INFO -   asle7a_tables.txt [table] → 1 chunks
2026-03-09 07:24:37,931 - INFO -   dostor_masr.txt [article] → 252 chunks
2026-03-09 07:24:37,933 - INFO -   drugs.txt [article] → 69 chunks
2026-03-09 07:24:37,933 - INFO -   drugs_tables.txt [table] → 1 chunks


2026-03-09 07:24:37,939 - INFO -   egra2at_gena2ya.txt [article] → 511 chunks
2026-03-09 07:24:37,941 - INFO -   erhab.txt [article] → 55 chunks
2026-03-09 07:24:37,942 - INFO -   قانون-الطوارئ_ultra_clean.txt [article] → 21 chunks
2026-03-09 07:24:37,943 - INFO -   tech.txt [article] → 46 chunks
2026-03-09 07:24:37,943 - INFO - ✓ laws total: 1570 chunks



PHASE 0: CHUNKING
  Total chunks: 1,570

PHASE 1: EXTRACTION SUMMARY
                            law_title                law_key                 law_id error  articles  penalties  definitions  references  topics  schedules
0                      قانون العقوبات             penal_code             penal_code  None       537         65            1         103      81          0
1            قانون مكافحة غسل الأموال       money_laundering       money_laundering  None        20          0            0          12       1          0
2              قانون الأسلحة والذخيرة     weapons_ammunition     weapons_ammunition  None        51          0            0          11      16         21
3                     الدستور الجنائي  criminal_constitution  criminal_constitution  None       251          0            0          15       2          0
4               قانون مكافحة المخدرات             anti_drugs             anti_drugs  None        68          1            0          26      40         18


2026-03-09 07:24:38,984 - INFO - ✓ Neo4j connected
2026-03-09 07:24:39,097 - INFO - ✓ Existing data cleared
2026-03-09 07:24:39,254 - INFO - Received notification from DBMS server: <GqlStatusObject gql_status='00NA0', status_description="note: successful completion - index or constraint already exists. The command 'CREATE CONSTRAINT law_id_unique IF NOT EXISTS FOR (e:Law) REQUIRE (e.law_id) IS UNIQUE' has no effect. The index or constraint specified by 'CONSTRAINT constraint_b9269926 FOR (e:Law) REQUIRE (e.law_id) IS UNIQUE' already exists.", position=None, raw_classification='SCHEMA', classification=<NotificationClassification.SCHEMA: 'SCHEMA'>, raw_severity='INFORMATION', severity=<NotificationSeverity.INFORMATION: 'INFORMATION'>, diagnostic_record={'_classification': 'SCHEMA', '_severity': 'INFORMATION', 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CREATE CONSTRAINT law_id_unique        IF NOT EXISTS FOR (l:Law)          REQUIRE l.law_id IS UNIQUE'
202


IMPORTING: قانون العقوبات
  ✓ 537 articles | 65 penalties | 0 schedules

IMPORTING: قانون مكافحة غسل الأموال
  ✓ 20 articles | 0 penalties | 0 schedules

IMPORTING: قانون الأسلحة والذخيرة
  ✓ 51 articles | 0 penalties | 21 schedules

IMPORTING: الدستور الجنائي
  ✓ 251 articles | 0 penalties | 0 schedules

IMPORTING: قانون مكافحة المخدرات
  ✓ 68 articles | 1 penalties | 18 schedules

IMPORTING: قانون الإجراءات الجنائية
  ✓ 510 articles | 16 penalties | 0 schedules

IMPORTING: قانون مكافحة الإرهاب
  ✓ 54 articles | 0 penalties | 0 schedules

IMPORTING: قانون الطوارئ
  ✓ 21 articles | 2 penalties | 0 schedules

IMPORTING: قانون مكافحة جرائم تقنية المعلومات


2026-03-09 07:29:04,279 - WARNING -   [SKIP] penal_code 36/2020-09-01 — no article numbers
2026-03-09 07:29:04,279 - INFO -   ✓ penal_code law=6/2020 type=modification articles=['392']


  ✓ 45 articles | 0 penalties | 0 schedules

PHASE 3: IMPORTING AMENDMENTS

  → penal_code: 1 amendment(s)
    ✓ law=6/2020 | type=modification | articles=['392']

  ✓ Total amendments imported: 1

✅ BUILD COMPLETE


2026-03-09 07:29:06,815 - WARNING - Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `DEFINES_SUBSTANCE` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=13, offset=12>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 12, 'line': 1, 'column': 13}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'MATCH ()-[r:DEFINES_SUBSTANCE]->() RETURN count(r) AS count'



KNOWLEDGE GRAPH STATISTICS

Nodes:
  Law: 9
  Article: 1558
  Penalty: 84
  Definition: 2
  Topic: 5
  Schedule: 11
  ScheduleEntry: 16
  Substance: 0
  Amendment: 1

Relationships:
  CONTAINS: 1558
  REFERENCES: 459
  HAS_PENALTY: 136
  DEFINES: 2
  TAGGED_WITH: 212
  HAS_SCHEDULE: 11
  CONTAINS_ENTRY: 16
  DEFINES_SUBSTANCE: 0
  HAS_AMENDMENT: 1
  AMENDED_BY: 1
  SUPERSEDES: 1

